# 03 · Chia train/val/test, tách seed → bài đích, tạo tracks.parquet (ticket #6, #7)

**Input** (Add Input): Kaggle Dataset `mixnext-processed` v1 (`sessions_raw.parquet`, `tracks_raw.parquet` từ notebook 02).

**Output** (`/kaggle/working/processed/`): `tracks.parquet`, `sessions.parquet`, `eval_targets.parquet`, `split_report.json`, kèm bản sao các file của v1, để thư mục này đủ để làm `mixnext-processed` v2.

Bật Internet để clone repo. Chỉ cần CPU.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #6, #7 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "a"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
from pathlib import Path
RAW_DIR = sorted(Path("/kaggle/input").rglob("sessions_raw.parquet"))[0].parent
OUT = Path("/kaggle/working/processed")
print("v1:", RAW_DIR, sorted(p.name for p in RAW_DIR.iterdir()))

In [ ]:
import json, shutil, time
import pandas as pd
from src.data.split import RANDOM_SEED, prepare, write_prepared

t0 = time.time()
sessions_raw = pd.read_parquet(RAW_DIR / "sessions_raw.parquet")
tracks_raw = pd.read_parquet(RAW_DIR / "tracks_raw.parquet")
result = prepare(sessions_raw, tracks_raw, RANDOM_SEED)   # có validate_dataset bên trong
paths = write_prepared(result, OUT)
for name in ("sessions_raw.parquet", "tracks_raw.parquet", "build_log.json"):
    if (RAW_DIR / name).exists():
        shutil.copy(RAW_DIR / name, OUT / name)
print(json.dumps(result.report, indent=2, ensure_ascii=False))
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.iterdir()):
    print(p.name, f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Chạy lại với cùng random_seed phải ra đúng cùng một tập (tiêu chí "Script chạy lại được")
def fingerprint(r):
    return tuple(int(pd.util.hash_pandas_object(df.astype({"seed_track_ids": str}) if "seed_track_ids" in df else df, index=False).sum())
                 for df in (r.tracks, r.sessions, r.eval_targets))
first = fingerprint(result)
del result
print("giống hệt lần đầu:", fingerprint(prepare(sessions_raw, tracks_raw, RANDOM_SEED)) == first)

## Kết quả (điền sau khi chạy)

Dán `split_report.json` ở trên vào mô tả PR.